In [1]:
%pylab inline

Populating the interactive namespace from numpy and matplotlib


In [2]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

In [3]:
import pandas as pd
from glob import glob
from tqdm import tqdm
from skimage import io
import deepdish as dd

In [12]:
from keras import backend as K
from keras.optimizers import Adam
from keras.regularizers import l2
from keras.models import Model, load_model
from keras.callbacks import ModelCheckpoint,EarlyStopping, ReduceLROnPlateau
from keras.layers import Input, BatchNormalization, Conv2D, Dense, Activation, Dropout

In [5]:
from keras.applications import Xception,VGG16,VGG19,ResNet50
from keras.applications import InceptionV3, InceptionResNetV2
from keras.applications import DenseNet121, DenseNet169, DenseNet201

variants = ['Xception', 'VGG16', 'VGG19', 
            'ResNet50', 'InceptionV3', 'InceptionResNetV2', 
            'DenseNet121', 'DenseNet169', 'DenseNet201']

In [6]:
angle_range = 30
rotation_step = 2
dropout_rate = 0.25

In [7]:
def get_features(features_file):
    
    features = dd.io.load(features_file)
    
    return(features)


def get_model(model_path, reset_weights=False):
    
    model = load_model(model_path)
    
    if reset_weights:
        reset_weights(model)
        
    return(model)

def reset_weights(model_in):
    
    session = K.get_session()
    
    for layer in model_in.layers: 
        if hasattr(layer, 'kernel_initializer'):
            layer.kernel.initializer.run(session=session)

In [8]:
def create_model(variant, dropout_rate=0.25):
    
    K.clear_session()

    if variant =='Xception':
        img_size = (139,139,3)
        model_pretrained = Xception(include_top=False,input_shape=img_size)

    elif variant =='VGG16':
        img_size = (139,139,3)
        model_pretrained = VGG16(include_top=False,input_shape=img_size)

    elif variant =='VGG19':
        img_size = (139,139,3)
        model_pretrained = VGG19(include_top=False,input_shape=img_size)

    elif variant =='ResNet50':
        img_size = (197,197,3)
        model_pretrained = ResNet50(include_top=False,input_shape=img_size)

    elif variant =='InceptionV3':
        img_size = (139,139,3)
        model_pretrained = InceptionV3(include_top=False,input_shape=img_size)

    elif variant =='InceptionResNetV2':
        img_size = (139,139,3)
        model_pretrained = InceptionResNetV2(include_top=False,input_shape=img_size)

    elif variant =='DenseNet121':
        img_size = (221,221,3)
        model_pretrained = DenseNet121(include_top=False,input_shape=img_size)

    elif variant =='DenseNet169':
        img_size = (221,221,3)
        model_pretrained = DenseNet169(include_top=False,input_shape=img_size)

    elif variant =='DenseNet201':
        img_size = (221,221,3)
        model_pretrained = DenseNet201(include_top=False,input_shape=img_size)
    
    return(model_pretrained)

In [9]:
def model_regressor(variant):
    
    model_pretrained = create_model(variant)
    model_pretrained._make_predict_function()
    
    feature_shape = model_pretrained.output_shape
    n = feature_shape[1]
    
    features_shape = model_pretrained.output_shape[1:]
    main_input = Input(shape=features_shape)

    x = main_input

    for i in range(n/2):
        
        r = x.shape.as_list()[1]
        if r>1:
            x = BatchNormalization()(x)
            x = Conv2D(filters=16*(i+1), 
                       kernel_size=(2,2), 
                       strides=(2,2),
                       kernel_regularizer=l2(l=1))(x)
            x = Dropout(dropout_rate)(x,training=True)

    x = BatchNormalization()(x)
    x = Dense(units=128, kernel_regularizer=l2(l=1))(x)
    x = Dropout(dropout_rate)(x,training=True)

    x = Dense(units=1, kernel_regularizer=l2(l=1))(x)
    main_output = Activation('relu')(x)

    model = Model(inputs=[main_input], outputs=[main_output])
    
    return(model)

In [19]:
models = {}
learning_rate = 1e-2

for variant in variants[:2]:
    
    print(variant)
    
    run_version = 'model_'+variant+'_'+str(angle_range)
    features_file = './01.features/model_'+variant+'_90_features.h5'
    model_file = './02.model/'+run_version+'.h5'

    features = get_features(features_file)
    
    data_x = features['data_x']
    data_y = features['data_y']
    data_paths = features['data_paths']
    data_rotation = features['data_rotation']

    subset = (np.absolute(data_rotation)<=60) & ((data_rotation%5)==0)

    train_x = data_x[subset]
    train_y = data_y[subset]
    data_paths = data_paths[subset]
    data_rotation = data_rotation[subset]
    
    
    MC = ModelCheckpoint(filepath='./02.model/new/'+run_version+'.h5')
    ES = EarlyStopping(patience=20)
    RL = ReduceLROnPlateau(factor=0.5, patience=5,min_lr=1e-6)

    
#    if variant in models.keys():
#        model = load_model(model_file)
    K.clear_session()
    model = model_regressor(variant)
    model.compile(optimizer=Adam(learning_rate), loss='mse')

    model.fit(x = train_x,
              y = train_y,
              validation_split=0.1,
              batch_size=25,
              callbacks=[ES, MC, RL],
              epochs=500,
              verbose=2)
    
    models[variant] = model

Xception
Train on 1890 samples, validate on 210 samples
Epoch 1/500
 - 6s - loss: 4091.1409 - val_loss: 1123.7804
Epoch 2/500
 - 5s - loss: 983.4290 - val_loss: 956.5723
Epoch 3/500
 - 5s - loss: 696.9061 - val_loss: 937.0139
Epoch 4/500
 - 5s - loss: 619.2040 - val_loss: 763.4269
Epoch 5/500
 - 5s - loss: 547.0131 - val_loss: 709.9051
Epoch 6/500
 - 5s - loss: 587.5714 - val_loss: 571.5445
Epoch 7/500
 - 5s - loss: 521.0853 - val_loss: 592.4041
Epoch 8/500
 - 5s - loss: 455.1176 - val_loss: 594.2569
Epoch 9/500
 - 5s - loss: 494.2863 - val_loss: 647.4107
Epoch 10/500
 - 5s - loss: 486.5628 - val_loss: 545.0595
Epoch 11/500
 - 5s - loss: 437.0427 - val_loss: 522.0671
Epoch 12/500
 - 5s - loss: 498.5599 - val_loss: 569.5033
Epoch 13/500
 - 5s - loss: 421.8020 - val_loss: 601.3372
Epoch 14/500
 - 5s - loss: 438.3258 - val_loss: 524.9163
Epoch 15/500
 - 5s - loss: 463.8470 - val_loss: 649.9241
Epoch 16/500
 - 5s - loss: 450.4535 - val_loss: 639.6566
Epoch 17/500
 - 5s - loss: 474.5491 - v

In [20]:
model.evaluate(train_x,train_y,batch_size=25)

2100/2100 [==============================] - 1s 346us/step


37.29113024757022